# Lab 1 — SOLUTION (instructor reference)
Fully worked clean-up of `hdb_messy.csv` with commentary on each decision.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/hdb_messy.csv")
df.info()  # 8 columns; resale_price is object; floor_area_sqm.1 duplicate; NaNs present

### 1. Price: strip `$` and comma, cast to float

In [ ]:
df["resale_price"] = (df["resale_price"].str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False).astype(float))
df["resale_price"].dtype  # float64

### 2. Dates: the format is mixed — parse with dayfirst for the dd/mm/yyyy rows

In [ ]:
df["sale_date"] = pd.to_datetime(df["sale_date"], format="mixed", dayfirst=True)
df["sale_date"].head()

### 3. Implausible values BEFORE imputation
9999, -50, 0 sqm are export bugs, not real flats. Treat as missing, then median-impute
(<1% of rows, roughly symmetric distribution → median is robust and simple).

In [ ]:
bad = ~df["floor_area_sqm"].between(30, 250)
df.loc[bad, "floor_area_sqm"] = np.nan
df["floor_area_sqm"] = df["floor_area_sqm"].fillna(df["floor_area_sqm"].median())
df["storey"] = df["storey"].fillna(df["storey"].median())
df[["floor_area_sqm", "storey"]].isna().sum()  # 0, 0

### 4. Duplicate column — note it has a *different name*, so `columns.duplicated()` won't catch it. Drop by name.

In [ ]:
df = df.drop(columns=["floor_area_sqm.1"])
assert "floor_area_sqm.1" not in df.columns

### 5. Final audit + export

In [ ]:
assert df["resale_price"].dtype == float
assert df["resale_price"].between(100000, 2000000).all()
assert df["sale_date"].notna().all()
df.to_csv("cleaned_hdb.csv", index=False)
df.describe()

**Teaching notes:** the most common learner errors — imputing *before* removing the 9999 outliers (median becomes garbage); using `errors='ignore'` in parsing (silently leaves bad values); forgetting the range sanity-check. The exit ticket asks which bug was hardest — expect 'the mixed dates' or 'the 9999 outlier'.